<b><font size="6"> Cars 4 You: Predicting Car Values with ML </font></b><br><br>

`Group 40`

Ana Macedo (20250405)<br>
Catarina Mendinhas (20250422)<br>
Lourenço Silva (20250453)<br>
Maria Fonseca (20250380)<br>

### <font color= '#BFD72F'>**Methodology** </font><a class="anchor" id='top'></a>

- [1. Abstract](#1)
- [2. Import Libraries](#2)
- [3. Metadata](#3)
- [4. Import Datasets](#4)
- [5. Feature Selection](#5)
    - [5.1 Filter Methods](#5_1)
        - [5.1.1 Removing Constant Features](#5_1_1)
        - [5.1.2 Correlation between features - Redundant Features](#5_1_2)
        - [5.1.3 Correlation with the target - Relevant Features](#5_1_3)
            - [5.1.3.1 Spearman Correlation](#5_1_3_1)
    - [5.2 Wrapped Methods](#5_2)
        - [5.2.1 RFE](#5_2_1)
    - [5.3 Embedded Methods](#5_3)
        - [5.3.1 Lasso](#5_3_1)
    - [5.4 Comparison between Models](#5_4)
- [6 Save the Data](#6)
- [7 End of The Notebook](#7)


<a class="anchor" id="1">

# **1. Abstract**

[Back to TOP](#TOP)
</a>

This project aims to develop a predictive model capable of estimating car prices based on their characteristics, using the Cars 4 You dataset. The goal is to support the company’s evaluation process by introducing an automated solution that accelerates car assessments and improves overall efficiency.

The current phase of the project focuses on feature selection, aiming to identify the most relevant features for predicting car prices. Different techniques were applied to evaluate the importance and contribution of each feature, reduce redundancy, and minimize noise in the data and improve signal.

In this notebook, many of the sections include helper functions created to streamline the implementation of feature selection techniques. These functions make it easier to apply the same process consistently across the training, test, and validation datasets.

<a class="anchor" id="2">

# **2. Import Libraries**

[Back to TOP](#TOP)
</a>

The following libraries will help us develop the analyses and model for this project. Pandas and Numpy, provide the efficient tools for data manipulation, cleaning and numerical computations. Matplotlib and Seaborn are used to create clear and informative visualizations. Scikit-learn offers a range of Machine Learning tools for model training, spliting the data and evaluate model performance. Finally, os, math and ceil are imported to support file management and mathematical operations.

In [1]:
import pandas as pd

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from math import ceil
import math


#filter methods
from sklearn.feature_selection import VarianceThreshold
from scipy.stats import spearmanr
# spearman 
from sklearn.feature_selection import SelectKBest, f_regression

# mutual information
from sklearn.feature_selection import mutual_info_classif

#wrapper methods
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVC
from sklearn.feature_selection import RFE


# embedded methods
from sklearn.linear_model import Lasso

# Load Created Functions
from feature_selection import *

# Set random seed for reproducibility
np.random.seed(40111) 


<a class="anchor" id="3">

# **3. Create Meta Data**

[Back to TOP](#TOP)
</a>

Understanding the features helps interpret the data correctly and supports subsequent analysis and modeling steps. This section provides a brief description of each feature in the dataset, explaining its meaning and type.

**carID:** An atribute that contains an identifier for each car.

**Brand:** The cars main brand (e.g., Ford, Toyota).

**model:** The car model.

**year:** The year of registration of the car.

**price:** The car's price when purchased by Cars 4 You (in £).

**transmission:** The car's type of transmission.

**mileage:** The total reported distance travelled by the car (in miles).

**fuelType:** Type of Fuel used by car (Diesel, Petrol, Hybrid, Electric).

**tax:** The amount of road tax (in £) that, in 2020, was applicable to the car in question.

**mpg:** Average Miles per Gallon.

**engineSize:** Size of Engine in liters (Cubic Decimeters).

**paintQuality%** The mechanic’s assessment of the cars’ overall paint quality and hull integrity (filled by the mechanic during evaluation). 

**previousOwner:** Number of previous registered owners of the vehicle.

**hasDamage:** Boolean marker filled by the seller at the time of registration stating whether the car is damaged or not.



<a class="anchor" id="4">

# **4. Import Dataset**

[Back to TOP](#TOP)
</a>

In this section, we load the preprocessed datasets that were previously saved after the data preprocessing steps. These datasets are ready for feature selection and further analysis.

In [2]:
# Define relative path to the preprocessed data folder (outside "notebooks/")
data_path = "../data_preprocessed/"

# Load preprocessed datasets
X_train = pd.read_csv(f"{data_path}X_train_preprocessed.csv", index_col=0)
X_val   = pd.read_csv(f"{data_path}X_val_preprocessed.csv", index_col=0)
test    = pd.read_csv(f"{data_path}test_preprocessed.csv", index_col=0)

y_train = pd.read_csv(f"{data_path}y_train.csv", index_col=0).squeeze()  # convert DataFrame → Series
y_val   = pd.read_csv(f"{data_path}y_val.csv", index_col=0).squeeze()

In [3]:
X_train.shape

(60778, 234)

<a class="anchor" id="5">

# **5. Feature Selection**

[Back to TOP](#TOP)
</a>

The feature selection pipeline we will implement has the following structure: First of all, we will remove the features gotten from the mechanic's evaluation. Then, we use filter methods to reduce noise by removing constant and redundant variables. After that, we used Spearman and ANOVA to see which features were most correlated with the target. Then we considered using RFE to further reduce dimensionality, but we found that it had a very high computational cost and opted to use the Elastic Net model instead.

Finally, we compared the Spearman, ANOVA and Elastic Net methods to make a final decision. We defined thresholds for each step so that we can change them and see if the model's performance improves.

<a class="anchor" id="5_1">

## **5.1 Remove Features that don't make sense**

[Back to TOP](#TOP)
</a>

Before anything else, we will drop the features related to the mechanic's evaluation: hasDamage and paintQuality%. This decision came from our EDA where we concluded that this information comes from the mechanic's evaluation and this ML project is replace this part of the process and thus function without this information.

In [4]:
X_train.shape

(60778, 234)

In [5]:
X_train = X_train.drop(['paintQuality%', 'hasDamage'], axis=1, errors='ignore')
X_val = X_val.drop(['paintQuality%', 'hasDamage'], axis=1, errors='ignore')

# Save a copy of X_train to use for fitting transformers only on training data with the initial features
X_train_fit_init = X_train.copy()

In [6]:
X_train.shape

(60778, 232)

<a class="anchor" id="5_1">

## **5.1 Filter Methods**

[Back to TOP](#TOP)
</a>

Filter methods are ideal for an initial screening of high-dimensional datasets due to their simplicity, even though they are considered less effective than model-based methods.
In this section, we will explore three filter-based techniques to select the features.

<a class="anchor" id="5_1_1">

### **5.1.1 Removing Constant Features**

[Back to TOP](#TOP)
</a>

This method identifies features that have low or zero variance, which are unlikely to contribute meaningful information to the model, and thus can help to reduce noise and improve efficiency.

In [7]:
# Used to fit transformers only on training data
X_train_fit = X_train.copy()

X_train = apply_variance_filter(X_train_fit, X_train, threshold=0.001, return_summary=True)
X_val = apply_variance_filter(X_train_fit, X_val, threshold=0.001, return_summary=False)

Total features kept: 130
Features selected: ['year', 'mileage', 'tax', 'mpg', 'engineSize', 'previousOwners', 'car_age', 'is_recent_car', 'is_hybrid_or_electric', 'is_automatic', 'fuel_efficiency_score', 'tax_to_engine_ratio', 'tax_efficiency', 'is_first_owner', 'brand_median_mileage', 'brand_avg_engineSize', 'model_median_tax', 'fueltype_avg_mpg', 'brand_avg_age', 'ohe_Brand_BMW', 'ohe_Brand_Ford', 'ohe_Brand_Hyundai', 'ohe_Brand_Mercedes', 'ohe_Brand_Opel', 'ohe_Brand_Skoda', 'ohe_Brand_Toyota', 'ohe_Brand_VW', 'ohe_model_ 2 Series', 'ohe_model_ 3 Series', 'ohe_model_ 4 Series', 'ohe_model_ 5 Series', 'ohe_model_ A Class', 'ohe_model_ A1', 'ohe_model_ A3', 'ohe_model_ A4', 'ohe_model_ A5', 'ohe_model_ A6', 'ohe_model_ A7', 'ohe_model_ A8', 'ohe_model_ Adam', 'ohe_model_ Amarok', 'ohe_model_ Arteon', 'ohe_model_ Astra', 'ohe_model_ Auris', 'ohe_model_ Avensis', 'ohe_model_ Aygo', 'ohe_model_ B Class', 'ohe_model_ B-MAX', 'ohe_model_ C Class', 'ohe_model_ C-HR', 'ohe_model_ C-MAX', 'oh

In [8]:
#Check the amount of features that were removed
X_train.shape

(60778, 130)

<a class="anchor" id="5_1_2">

### **5.1.2 Correlation between features - Redundant Features**

[Back to TOP](#TOP)
</a>

We use correlation measures to identify highly correlated features. Removing redundant features helps reduce multicollinearity and simplifies the dataset, making the model more efficient and easier to interpret.


We decided on the threshold based on the results above and applied it to the original datasets.

In [9]:
# Create a copy of X_train to fit the following method
X_train_fit = X_train.copy()

X_train = remove_highly_correlated_features(X_train_fit, X_train, threshold=0.95, return_summary=True)
X_val = remove_highly_correlated_features(X_train_fit, X_val, threshold=0.95, return_summary=False)

Total features kept: 126
Features selected: ['year', 'mileage', 'tax', 'mpg', 'engineSize', 'previousOwners', 'is_recent_car', 'is_hybrid_or_electric', 'is_automatic', 'fuel_efficiency_score', 'tax_to_engine_ratio', 'tax_efficiency', 'is_first_owner', 'brand_median_mileage', 'brand_avg_engineSize', 'model_median_tax', 'fueltype_avg_mpg', 'brand_avg_age', 'ohe_Brand_BMW', 'ohe_Brand_Ford', 'ohe_Brand_Hyundai', 'ohe_Brand_Mercedes', 'ohe_Brand_Opel', 'ohe_Brand_Skoda', 'ohe_Brand_Toyota', 'ohe_Brand_VW', 'ohe_model_ 2 Series', 'ohe_model_ 3 Series', 'ohe_model_ 4 Series', 'ohe_model_ 5 Series', 'ohe_model_ A Class', 'ohe_model_ A1', 'ohe_model_ A3', 'ohe_model_ A4', 'ohe_model_ A5', 'ohe_model_ A6', 'ohe_model_ A7', 'ohe_model_ A8', 'ohe_model_ Adam', 'ohe_model_ Amarok', 'ohe_model_ Arteon', 'ohe_model_ Astra', 'ohe_model_ Auris', 'ohe_model_ Avensis', 'ohe_model_ Aygo', 'ohe_model_ B Class', 'ohe_model_ B-MAX', 'ohe_model_ C Class', 'ohe_model_ C-HR', 'ohe_model_ C-MAX', 'ohe_model_ CL

In [10]:
X_train

,year,mileage,tax,mpg,engineSize,previousOwners,is_recent_car,is_hybrid_or_electric,is_automatic,fuel_efficiency_score,...,ohe_model_ Yaris,ohe_model_ Yeti,ohe_model_ Yeti Outdoor,ohe_model_ Z4,ohe_model_ Zafira,ohe_transmission_Semi-Auto,ohe_transmission_unknown,ohe_fuelType_Other,ohe_mileage_category_Medium,ohe_mileage_category_Very Low
carID,,,,,,,,,,,,,,,,,,,,,
59459,2016.0,29601.0,81.0,62.8,1.0,2.0,0.0,0.0,0.0,62.800000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
49082,2017.0,62140.0,145.0,64.2,2.0,3.0,1.0,0.0,0.0,32.100000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
14425,2008.0,87470.8,189.0,42.2,2.0,1.0,0.0,0.0,1.0,21.100000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
13190,2019.0,11208.0,145.0,52.3,2.0,4.0,1.0,0.0,1.0,26.150000,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
60655,2019.0,2357.0,145.0,44.8,1.2,3.0,1.0,0.0,0.0,37.333333,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47192,2019.0,4797.0,145.0,62.8,1.6,1.0,1.0,0.0,1.0,39.250000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
71813,2012.0,87470.8,189.0,47.7,2.0,3.0,0.0,0.0,1.0,23.850000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4910,2019.0,1800.0,145.0,47.9,1.0,2.0,1.0,0.0,0.0,47.900000,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [11]:
# Check that some features were removed
X_train.shape

(60778, 126)

The application of initial Filter Methods removing constant and highly correlated features has refined our dataset to 130 features.

The subsequent phase involves assessing the predictive relevance of these remaining 130 features against the target. So, we will now compare the results obtained from advanced Filter Methods (measuring relevance through statistical significance) with the model-based insights provided by Embedded Methods specifically, Elastic Net.

<a class="anchor" id="5_1_3">

### **5.1.3 Correlation with the target - Relevant Features**

[Back to TOP](#TOP)
</a>

These processes consist of setting a correlation threshold and keeping only the features that show a meaningful relationship with the target feature. By doing so, we aim to remove irrelevant or redundant features, simplifying the dataset and improving the efficiency of the subsequent modeling phase.

<a class="anchor" id="5_1_3_1">

#### **5.1.3.1 Spearman Correlation**

[Back to TOP](#TOP)
</a>

We decided to utilize Spearman correlation because it measures monotonic relationships rather than being strictly limited to linear relationships, as is the case with the Pearson coefficient. This is crucial because, in the context of car pricing, the depreciation (for example mileage) or the impact of other characteristics is often non-linear. Additionally, Spearman is based on rankings, which makes it significantly more robust to outliers, providing a more reliable measure of feature relevance for our dataset.

In [12]:
# Create a copy of X_train to fit the following methods
X_train_fit = X_train.copy()

X_train = spearman_correlation_selection(X_train_fit, X_train, y_train, threshold=0.3, return_summary=True)
X_val = spearman_correlation_selection(X_train_fit, X_val, y_train, threshold=0.3, return_summary=False)

Total features kept: 12
Features selected by Spearman method: ['year', 'is_automatic', 'fuel_efficiency_score', 'engineSize', 'mileage', 'ohe_mileage_category_Very Low', 'is_recent_car', 'ohe_transmission_Semi-Auto', 'tax_efficiency', 'mpg', 'ohe_Brand_Mercedes', 'ohe_Brand_Opel']
Nº Features eliminated: 114


In [13]:
# Save the list of selected features with Filter Methods to use later for comparison
filter_cols = X_train.columns.tolist()

<a class="anchor" id="5_2">

## **5.2 Wrapped Methods**

[Back to TOP](#TOP)
</a>

<a class="anchor" id="5_2_1">

### **5.2.1 RFE**

[Back to TOP](#TOP)
</a>

In [14]:
wrapped_cols = rfe_selection(X_train_fit_init, X_train, y_train, LinearRegression(), n_features=30, return_summary=True)

ValueError: Input X contains NaN.
LinearRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

<a class="anchor" id="5_3">

## **5.3 Embedded Methods**

[Back to TOP](#TOP)
</a>

<a class="anchor" id="5_3_1">

### **5.3.1 Lasso**

[Back to TOP](#TOP)
</a>

We predicted that Lasso Regression (L1 Regularization) would overperform over Ridge (L2) because of its unique capability for automatic feature selection, but decided to test it anyway using ElasticNetCV.

In [ ]:
embedded_cols = lasso_selection(X_train_fit_init, X_train, y_train, threshold=0.01, return_summary=True)

Total features kept: 114
Features selected by Lasso method: ['year', 'mileage', 'tax', 'mpg', 'engineSize', 'previousOwners', 'is_recent_car', 'is_hybrid_or_electric', 'is_automatic', 'fuel_efficiency_score', 'tax_to_engine_ratio', 'tax_efficiency', 'is_first_owner', 'brand_median_mileage', 'brand_avg_engineSize', 'model_median_tax', 'fueltype_avg_mpg', 'brand_avg_age', 'ohe_Brand_BMW', 'ohe_Brand_Hyundai', 'ohe_Brand_Opel', 'ohe_Brand_Toyota', 'ohe_Brand_VW', 'ohe_model_ 2 Series', 'ohe_model_ 3 Series', 'ohe_model_ 4 Series', 'ohe_model_ 5 Series', 'ohe_model_ A Class', 'ohe_model_ A1', 'ohe_model_ A3', 'ohe_model_ A4', 'ohe_model_ A5', 'ohe_model_ A6', 'ohe_model_ A7', 'ohe_model_ Adam', 'ohe_model_ Amarok', 'ohe_model_ Arteon', 'ohe_model_ Astra', 'ohe_model_ Auris', 'ohe_model_ Avensis', 'ohe_model_ Aygo', 'ohe_model_ B Class', 'ohe_model_ B-MAX', 'ohe_model_ C Class', 'ohe_model_ C-HR', 'ohe_model_ Citigo', 'ohe_model_ Combo Life', 'ohe_model_ Corolla', 'ohe_model_ Corsa', 'ohe_m

Our predictions were confirmed (L1 Ratio = 1) and we will use Lasso as our primary Embedded Method.

While Ridge only shrinks coefficients close to zero (retaining all features), the Lasso penalty forces the coefficients of irrelevant features to be exactly zero, allowing it to perform true feature elimination. This is ideal for our high-dimensional, post-encoded dataset, as it provides an efficient and rigorous method for reducing the 130 remaining features to a cleaner subset for model training.

Our cross‑validation chose alpha ≈ 5.99 as the value that best balances bias and variance for this problem. Given the scale of our features and target, a relatively strong L1 regularization is beneficial. Weaker penalties (smaller alpha) would overfit, and stronger ones (larger alpha) would throw away too much signal.

When comparing these coefficients with the results gotten from the Spearman and ANOVA analysis, we see a few **contradictions**. For example, we previously found that the features "tax" and "Brand_Mercedes" were among the features that better helped explain the model, but Lasso gave them a coefficient = 0, effectively removing their contribution.

This might be explained because Spearman and ANOVA look at each feature in isolation and ask how strongly it is related to the target. Lasso, on the other hand, fits a model with all features together and asks whether each feature still improves prediction once the others are already in the model. 

If **“tax” or “Brand_Mercedes” are strongly correlated with other variables** (for example, other price‑related or brand‑related features), or if their extra contribution is small, Lasso can set their coefficients to zero even though they are individually well correlated with the target. In practice, that means those features carry signal, but **most of that signal is already captured by other predictors**, so their unique added value is small or redundant.

<a class="anchor" id="5_4">

## **5.4 Comparison between Models**

[Back to TOP](#TOP)
</a>

As we mentioned previously, we will construct a table that allows us to compare the decisions of the different selection methods we used and identify consistently important features for our regression model.

In [ ]:
comparison_feature_selection(X_train, filter_cols, wrapped_cols, embedded_cols, n_agreed=2, return_summary=True)

Total features before selection: 126
Total features selected by at least 2 methods: 39
Selected features: ['year', 'mileage', 'mpg', 'engineSize', 'is_recent_car', 'is_hybrid_or_electric', 'is_automatic', 'fuel_efficiency_score', 'tax_efficiency', 'brand_avg_engineSize', 'brand_avg_age', 'ohe_Brand_BMW', 'ohe_Brand_Opel', 'ohe_model_ Adam', 'ohe_model_ Corsa', 'ohe_model_ Edge', 'ohe_model_ GLC Class', 'ohe_model_ I800', 'ohe_model_ Ka+', 'ohe_model_ Kodiaq', 'ohe_model_ Kona', 'ohe_model_ M4', 'ohe_model_ Meriva', 'ohe_model_ Q7', 'ohe_model_ RS6', 'ohe_model_ SL CLASS', 'ohe_model_ SLK', 'ohe_model_ Santa Fe', 'ohe_model_ Touareg', 'ohe_model_ Tucson', 'ohe_model_ Up', 'ohe_model_ V Class', 'ohe_model_ X3', 'ohe_model_ X4', 'ohe_model_ X5', 'ohe_model_ Yaris', 'ohe_transmission_Semi-Auto', 'ohe_mileage_category_Medium', 'ohe_mileage_category_Very Low']


['year',
 'mileage',
 'mpg',
 'engineSize',
 'is_recent_car',
 'is_hybrid_or_electric',
 'is_automatic',
 'fuel_efficiency_score',
 'tax_efficiency',
 'brand_avg_engineSize',
 'brand_avg_age',
 'ohe_Brand_BMW',
 'ohe_Brand_Opel',
 'ohe_model_ Adam',
 'ohe_model_ Corsa',
 'ohe_model_ Edge',
 'ohe_model_ GLC Class',
 'ohe_model_ I800',
 'ohe_model_ Ka+',
 'ohe_model_ Kodiaq',
 'ohe_model_ Kona',
 'ohe_model_ M4',
 'ohe_model_ Meriva',
 'ohe_model_ Q7',
 'ohe_model_ RS6',
 'ohe_model_ SL CLASS',
 'ohe_model_ SLK',
 'ohe_model_ Santa Fe',
 'ohe_model_ Touareg',
 'ohe_model_ Tucson',
 'ohe_model_ Up',
 'ohe_model_ V Class',
 'ohe_model_ X3',
 'ohe_model_ X4',
 'ohe_model_ X5',
 'ohe_model_ Yaris',
 'ohe_transmission_Semi-Auto',
 'ohe_mileage_category_Medium',
 'ohe_mileage_category_Very Low']

The resulting table shows, for each feature, whether it was selected (1) or eliminated (0) by each feature selection method: F-Regression, Elastic Net, and Spearman correlation.

A higher Sum value means more agreement between methods, suggesting that the feature is likely more important for predicting the target. Features with Sum = 0 were not selected by any method, indicating they are likely irrelevant.

Also based on this results, we can see a total of 130 features, but to be more conservative **we will keep only the features that were selected by all three methods (Sum= 3)**. This ensures that we retain only the most consistently important features.

In [ ]:
X_train_final.shape

(60778, 20)

In [ ]:
X_train_final.columns

Index(['year', 'engineSize', 'Brand_Hyundai', 'Brand_Opel', 'model_ Fabia',
       'model_ GLC Class', 'model_ Ka+', 'model_ M4', 'model_ Q7',
       'model_ RS6', 'model_ SL CLASS', 'model_ Santa Fe', 'model_ Up',
       'model_ V Class', 'model_ X3', 'model_ X4', 'model_ X5', 'model_ Yaris',
       'fuelType_Hybrid', 'mileage_category_Very Low'],
      dtype='object')

<a class="anchor" id="6">

# **6 Save the Data**

[Back to TOP](#TOP)
</a>

This section saves the processed dataset containing only the selected features, ensuring it is ready to be used in the model assessment notebook.

In [ ]:
# Go one level up from the notebooks folder to reach the repo root
selected_dir = "../data_feature_selected"
os.makedirs(selected_dir, exist_ok=True)

# Save processed datasets as CSV files
X_train_final.to_csv(f"{selected_dir}/X_train_final.csv", index=True)
X_val_final.to_csv(f"{selected_dir}/X_val_final.csv", index=True)
test_final.to_csv(f"{selected_dir}/test_final.csv", index=True)

y_train.to_csv(f"{selected_dir}/y_train.csv", index=True)
y_val.to_csv(f"{selected_dir}/y_val.csv", index=True)

<a class="anchor" id="7">

# **7 End of the Notebook**

[Back to TOP](#TOP)
</a>

The feature selection process refined the dataset and enhancing the overall performance of the predictive model. Several complementary techniques were applied to identify the most relevant features, reduce redundancy, and remove irrelevant features. This systematic approach helped ensure that the final set of features captured the most meaningful relationships with the target feature, improving both model efficiency and interpretability. Additionally, the dataset’s shape was continuously monitored after each step to confirm that all transformations were correctly applied and that the data remained consistent throughout the process.